# Amharic Speech-to-Speech Assistant

Clean ASR → LLM → TTS pipeline.

**Models**
- ASR: `snapwre/hohe-asr-amharic`
- LLM: `google/gemma-4-E2B-it`
- TTS: `african-low-resource/omnivoice-amharic`

The pipeline uses GPU time-slicing so Gemma generates the first stable phrase, OmniVoice synthesizes that phrase, and Gemma then continues the response.

## 1. Environment Setup

Use a Colab GPU runtime. The versions below match the working pipeline configuration.

In [ ]:
!nvidia-smi
!pip install -q "transformers==5.18.0" accelerate sentencepiece librosa gradio soundfile pydub tensorboardX webdataset
!pip install -q --no-deps omnivoice==0.2.1

In [ ]:
import re
import time
import uuid

import librosa
import soundfile as sf
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Hugging Face Authentication

Gemma requires access to the model repository. Authenticate with a Hugging Face account that has accepted the model terms.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 3. Load Gemma 4 E2B-it

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

LLM_MODEL_ID = "google/gemma-4-E2B-it"

llm_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

processor = AutoProcessor.from_pretrained(
    LLM_MODEL_ID
)

model = AutoModelForMultimodalLM.from_pretrained(
    LLM_MODEL_ID,
    dtype=llm_dtype,
    device_map="auto",
    attn_implementation="sdpa",
).eval()

SYSTEM = """
You are a helpful voice assistant.

Rules:
- Understand the user's request carefully.
- If the user speaks Amharic, respond naturally in Amharic.
- Give a direct answer rather than repeating the user's question.
- Keep the answer short and suitable for spoken conversation.
- Do not invent information.
"""

print("Gemma loaded")
print("Device:", next(model.parameters()).device)
print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

## 4. Load Hohe Amharic ASR

In [ ]:
from transformers import AutoProcessor as ASRAutoProcessor
from transformers import AutoModelForCTC

ASR_MODEL_ID = "snapwre/hohe-asr-amharic"

asr_processor = ASRAutoProcessor.from_pretrained(
    ASR_MODEL_ID
)

asr_dtype = torch.float16

asr_model = AutoModelForCTC.from_pretrained(
    ASR_MODEL_ID,
    torch_dtype=asr_dtype,
    low_cpu_mem_usage=True,
).to("cuda").eval()

print("Hohe loaded")
print("ASR device:", next(asr_model.parameters()).device)
print("ASR dtype:", next(asr_model.parameters()).dtype)
print(
    "Total GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

In [ ]:
def clean_hohe_text(text):
    text = re.sub(
        r"^\s*\[AMH\]\s*",
        "",
        text,
    )
    return text.strip()


def transcribe_hohe(audio_path):
    total_start = time.perf_counter()

    audio, _ = librosa.load(
        audio_path,
        sr=16000,
        mono=True,
    )

    inputs = asr_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    gpu_inputs = {}

    for key, value in inputs.items():
        if value.is_floating_point():
            gpu_inputs[key] = value.to(
                "cuda",
                dtype=asr_dtype,
            )
        else:
            gpu_inputs[key] = value.to("cuda")

    prep_end = time.perf_counter()

    torch.cuda.synchronize()
    infer_start = time.perf_counter()

    with torch.inference_mode():
        logits = asr_model(
            **gpu_inputs
        ).logits

    torch.cuda.synchronize()
    infer_end = time.perf_counter()

    pred_ids = logits.argmax(dim=-1)

    raw_text = asr_processor.batch_decode(
        pred_ids
    )[0]

    transcript = clean_hohe_text(
        raw_text
    )

    total_end = time.perf_counter()

    return {
        "raw": raw_text,
        "transcript": transcript,
        "preprocessing": prep_end - total_start,
        "inference": infer_end - infer_start,
        "total": total_end - total_start,
    }

In [ ]:
import numpy as np

dummy_audio = np.zeros(
    16000,
    dtype=np.float32,
)

dummy_inputs = asr_processor(
    dummy_audio,
    sampling_rate=16000,
    return_tensors="pt",
)

dummy_gpu = {}

for key, value in dummy_inputs.items():
    if value.is_floating_point():
        dummy_gpu[key] = value.to(
            "cuda",
            dtype=asr_dtype,
        )
    else:
        dummy_gpu[key] = value.to("cuda")

with torch.inference_mode():
    _ = asr_model(**dummy_gpu).logits

torch.cuda.synchronize()

print("Hohe warm-up complete")

## 5. Load OmniVoice Amharic TTS

In [ ]:
from omnivoice import OmniVoice

try:
    from omnivoice import OmniVoiceGenerationConfig
except ImportError:
    from omnivoice.models.omnivoice import OmniVoiceGenerationConfig


TTS_MODEL_ID = "african-low-resource/omnivoice-amharic"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tts_model = OmniVoice.from_pretrained(
    TTS_MODEL_ID,
    device_map="cuda:0",
    dtype=torch.float16,
)

tts_config_8 = OmniVoiceGenerationConfig(
    num_step=8,
    guidance_scale=2.0,
)

print("OmniVoice Amharic loaded")
print(
    "Total GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

## 6. Time-Sliced LLM and TTS

Gemma stops after its first stable phrase. OmniVoice synthesizes that phrase while Gemma is paused, then Gemma continues generation. This avoids simultaneous Gemma and OmniVoice GPU workloads.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList


PHRASE_ENDINGS = (
    "፣",
    "።",
    "፤",
    "፡፡",
    ".",
    "!",
    "?",
)


def get_stable_phrase(
    text,
    min_words=2,
):
    text = text.strip()

    for i, ch in enumerate(text):
        if ch in PHRASE_ENDINGS:
            candidate = text[:i + 1].strip()

            if len(candidate.split()) >= min_words:
                return candidate

    if "፡፡" in text:
        end = text.find("፡፡") + 2
        candidate = text[:end].strip()

        if len(candidate.split()) >= min_words:
            return candidate

    return None


class StopAtStablePhrase(StoppingCriteria):

    def __init__(
        self,
        tokenizer,
        prompt_length,
        min_words=2,
    ):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.min_words = min_words

    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):
        generated = input_ids[
            0,
            self.prompt_length:
        ]

        text = self.tokenizer.decode(
            generated,
            skip_special_tokens=True,
        )

        return (
            get_stable_phrase(
                text,
                min_words=self.min_words,
            )
            is not None
        )

In [ ]:
def generate_first_phrase(user_text):

    messages = [
        {
            "role": "system",
            "content": SYSTEM,
        },
        {
            "role": "user",
            "content": user_text,
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

    prompt_length = inputs[
        "input_ids"
    ].shape[-1]

    stopper = StopAtStablePhrase(
        tokenizer=processor.tokenizer,
        prompt_length=prompt_length,
        min_words=2,
    )

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            use_cache=True,
            stopping_criteria=StoppingCriteriaList(
                [stopper]
            ),
        )

    torch.cuda.synchronize()
    end = time.perf_counter()

    generated_ids = output[
        0,
        prompt_length:
    ]

    text = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    phrase = get_stable_phrase(
        text,
        min_words=2,
    )

    if phrase is None:
        phrase = text

    return {
        "inputs": inputs,
        "output_ids": output,
        "prompt_length": prompt_length,
        "phrase": phrase,
        "stage_text": text,
        "time": end - start,
    }


def synthesize_first_chunk(text):

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        audio = tts_model.generate(
            text=text,
            language="Amharic",
            generation_config=tts_config_8,
        )

    torch.cuda.synchronize()

    waveform = audio[0]

    output_path = (
        f"/content/timeslice_"
        f"{uuid.uuid4().hex[:8]}.wav"
    )

    sf.write(
        output_path,
        waveform,
        tts_model.sampling_rate,
    )

    end = time.perf_counter()

    duration = (
        len(waveform)
        / tts_model.sampling_rate
    )

    return {
        "path": output_path,
        "time": end - start,
        "duration": duration,
    }


def continue_gemma(
    first_stage,
    max_new_tokens=48,
):

    stage_ids = first_stage[
        "output_ids"
    ]

    original_inputs = first_stage[
        "inputs"
    ]

    prompt_length = first_stage[
        "prompt_length"
    ]

    continuation_inputs = {
        "input_ids": stage_ids,
        "attention_mask": torch.ones_like(
            stage_ids
        ),
    }

    if "token_type_ids" in original_inputs:
        original_types = original_inputs[
            "token_type_ids"
        ]

        extra_length = (
            stage_ids.shape[-1]
            - original_types.shape[-1]
        )

        extra_types = original_types[
            :, -1:
        ].expand(
            -1,
            extra_length,
        )

        continuation_inputs[
            "token_type_ids"
        ] = torch.cat(
            [
                original_types,
                extra_types,
            ],
            dim=-1,
        )

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        completed = model.generate(
            **continuation_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    end = time.perf_counter()

    full_generated = completed[
        0,
        prompt_length:
    ]

    full_text = processor.tokenizer.decode(
        full_generated,
        skip_special_tokens=True,
    ).strip()

    return {
        "text": full_text,
        "time": end - start,
    }

## 7. End-to-End ASR → LLM → TTS Pipeline

In [ ]:
def run_timesliced_s2s(audio_path):

    torch.cuda.reset_peak_memory_stats()

    overall_start = time.perf_counter()

    asr = transcribe_hohe(
        audio_path
    )

    if not asr["transcript"]:
        raise ValueError(
            "No speech was transcribed."
        )

    first_stage = generate_first_phrase(
        asr["transcript"]
    )

    if not first_stage["phrase"]:
        raise ValueError(
            "Gemma did not produce a usable first phrase."
        )

    first_audio = synthesize_first_chunk(
        first_stage["phrase"]
    )

    first_audio_ready_time = (
        time.perf_counter()
        - overall_start
    )

    continuation = continue_gemma(
        first_stage
    )

    total_compute = (
        time.perf_counter()
        - overall_start
    )

    return {
        "transcript":
            asr["transcript"],

        "first_phrase":
            first_stage["phrase"],

        "full_response":
            continuation["text"],

        "audio_path":
            first_audio["path"],

        "asr":
            asr["total"],

        "first_phrase_generation":
            first_stage["time"],

        "first_phrase_tts":
            first_audio["time"],

        "first_audio_ready":
            first_audio_ready_time,

        "first_audio_duration":
            first_audio["duration"],

        "continuation_generation":
            continuation["time"],

        "total_compute":
            total_compute,

        "peak_vram":
            torch.cuda.max_memory_allocated()
            / 1024**3,
    }

## 8. Gradio Voice Demo

The audio output is the first synthesized response chunk. The complete Gemma response is shown as text.

In [ ]:
import gradio as gr


def voice_assistant(audio_path):

    if audio_path is None:
        return (
            "",
            "",
            "",
            None,
            "No audio received.",
        )

    try:
        result = run_timesliced_s2s(
            audio_path
        )

        metrics = (
            f"ASR: {result['asr']:.3f} s\n"
            f"First phrase generation: "
            f"{result['first_phrase_generation']:.3f} s\n"
            f"First phrase TTS: "
            f"{result['first_phrase_tts']:.3f} s\n"
            f"FIRST AUDIO READY: "
            f"{result['first_audio_ready']:.3f} s\n"
            f"First audio duration: "
            f"{result['first_audio_duration']:.3f} s\n"
            f"Continuation generation: "
            f"{result['continuation_generation']:.3f} s\n"
            f"Total compute: "
            f"{result['total_compute']:.3f} s\n"
            f"Peak VRAM: "
            f"{result['peak_vram']:.2f} GB"
        )

        return (
            result["transcript"],
            result["first_phrase"],
            result["full_response"],
            result["audio_path"],
            metrics,
        )

    except Exception as e:
        return (
            "",
            "",
            "",
            None,
            f"ERROR:\n{type(e).__name__}: {e}",
        )


with gr.Blocks() as demo:

    gr.Markdown(
        "# Amharic Speech-to-Speech Assistant"
    )

    input_audio = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="Speak in Amharic",
    )

    run_button = gr.Button(
        "Run"
    )

    transcript_box = gr.Textbox(
        label="Hohe ASR Transcript"
    )

    phrase_box = gr.Textbox(
        label="First Stable Phrase"
    )

    response_box = gr.Textbox(
        label="Full Gemma Response"
    )

    output_audio = gr.Audio(
        label="First OmniVoice Audio Chunk",
        autoplay=True,
    )

    metrics_box = gr.Textbox(
        label="Latency"
    )

    run_button.click(
        voice_assistant,
        inputs=input_audio,
        outputs=[
            transcript_box,
            phrase_box,
            response_box,
            output_audio,
            metrics_box,
        ],
    )


demo.launch(share=True)